In [1]:
import polars as pl
import numpy as np
import os
import re
import random
from utils_tools.data_preprocessing.data_deal import DataProcessor


#self.df,self.all_columns,self.sn_na_list=DataProcessor(df,all_columns).main()

def allocate_sn(csv_path: str, per_total: int, output_path: str, seed: int = None):
    """
    根据每个 sn_prefix 的比例，从原始数据中随机抽取对应数量的 sn；
    如果最终发现分配数量不足 per_total，则从剩余的未选中行中再次补足。

    :param csv_path: 原始 sn 数据的 CSV 文件路径。
    :param per_total: 需要分配的总 sn 数量。
    :param output_path: 分配结果保存的 CSV 文件路径。
    :param seed: 随机种子（可选），用于结果可重复。
    """
    # 0. 设置随机种子（可选）
    if seed is not None:
        np.random.seed(seed)
    
    # 1. 读取 CSV 文件
    ok_sn = pl.read_csv(csv_path)
    
    # 2. 提取 'sn' 的前6个字符作为 'sn_prefix'
    ok_sn = ok_sn.with_columns(
        pl.col('sn').str.slice(0, 6).alias('sn_prefix')
    )

    # 3. 按 'sn_prefix' 分组并计数
    sn_counts = ok_sn.groupby('sn_prefix').agg(
        pl.count().alias('count')
    )

    # 4. 获取总行数
    total_count = ok_sn.height

    # 5. 计算每个 'sn_prefix' 的比例
    sn_proportions = sn_counts.with_columns(
        (pl.col('count') / total_count).alias('proportion')
    )

    # 6. 计算每种 'sn_prefix' 的精确需求数量
    sn_proportions = sn_proportions.with_columns(
        (pl.col('proportion') * per_total).alias('exact_required')
    )

    # 7. 分配 floor 值并记录小数部分
    sn_proportions = sn_proportions.with_columns([
        pl.col('exact_required').floor().cast(pl.Int64).alias('required_count'),
        (pl.col('exact_required') - pl.col('exact_required').floor()).alias('fraction')
    ])

    # 8. 计算当前总分配数和剩余数
    current_total = sn_proportions.select(pl.sum('required_count')).item()
    remainder = per_total - current_total

    # 根据 remainder，将一些前缀 +1 或 -1
    if remainder > 0:
        # 按 fraction 从大到小排序，给前 remainder 个 sn_prefix +1
        #sn_proportions = sn_proportions.sort('fraction', reverse=True)
        sn_proportions = sn_proportions.with_columns(
            pl.when(pl.arange(0, pl.count()) < remainder)
              .then(pl.col('required_count') + 1)
              .otherwise(pl.col('required_count'))
              .alias('required_count')
        )
    elif remainder < 0:
        # 按 fraction 从小到大排序，给前 abs(remainder) 个 -1
        sn_proportions = sn_proportions.sort('fraction', reverse=False)
        sn_proportions = sn_proportions.with_columns(
            pl.when(pl.arange(0, pl.count()) < abs(remainder))
              .then(pl.when(pl.col('required_count') > 0).then(pl.col('required_count') - 1).otherwise(0))
              .otherwise(pl.col('required_count'))
              .alias('required_count')
        )

    # 确保每个前缀的分配不超过其可用数量
    sn_proportions = sn_proportions.with_columns(
        pl.min_horizontal([pl.col('required_count'), pl.col('count')]).alias('required_count')
    )

    # 重新计算“理论”分配总数（可能由于可用数量限制导致不足）
    final_assigned = sn_proportions.select(pl.sum('required_count')).item()

    # 9. 初次抽取（根据 required_count 分配）
    #    先把分配表与 ok_sn 合并
    sn_full = ok_sn.join(
        sn_proportions.select(['sn_prefix', 'required_count']),
        on='sn_prefix',
        how='left'
    )

    # 给 sn_full 加随机数
    random_values = np.random.rand(sn_full.height)
    sn_full = sn_full.with_columns(
        pl.Series('rand', random_values)
    )

    # 按 sn_prefix, rand 排序
    sn_full = sn_full.sort(['sn_prefix', 'rand'])

    # 为每条记录加一个在各自 prefix 组内的累加行号 row_num
    sn_full = sn_full.with_columns(
        pl.arange(0, sn_full.height).alias('row_id')
    ).with_columns(
        pl.col('row_id').cumcount().over('sn_prefix').alias('row_num')
    )

    # 先选出 row_num < required_count 的行，作为“已分配”
    sn_selected = sn_full.filter(
        pl.col('row_num') < pl.col('required_count')
    )

    # 10. 如果 final_assigned 仍不足 per_total，则从未选中的行再随机补足
    #     注意：这里的 final_assigned 只是“理论上”的值，实际可能还达不到
    #     我们以 sn_selected.height 为准来比较
    actual_assigned = sn_selected.height
    if actual_assigned < per_total:
        short = per_total - actual_assigned

        # 找到“未选中的行”
        sn_unselected = sn_full.filter(
            pl.col('row_num') >= pl.col('required_count')
        )

        if sn_unselected.height < short:
            raise ValueError(
                f"可用于补足的 SN 不足，尚需要 {short}，"
                f"但只剩 {sn_unselected.height} 可用。"
            )
        
        # 从未分配的行中随机选 short 个
        extra_indices = np.random.choice(sn_unselected.height, size=short, replace=False)
        sn_extra = sn_unselected[extra_indices]

        # 将额外选出的行与原先已选的合并
        sn_selected = pl.concat([sn_selected, sn_extra])

    # 最终分配后的数量
    final_total = sn_selected.height
    if final_total != per_total:
        raise ValueError(
            f"分配后的总数 {final_total} 不等于设定值 {per_total}，请检查数据或调整逻辑。"
        )

    # 保留主要字段
    sn_selected = sn_selected.select(['sn_prefix', 'sn'])

    # 打印结果
    print("分配完成，每种 sn_prefix 分配的具体 sn 行如下：")
    print(sn_selected)

    # 保存结果到新的 CSV 文件
    sn_selected.write_csv(output_path)
    print(f"分配结果已保存到: {output_path}")
    return sn_selected
#
def get_train_sn(csv_path,per_total,seed):
    sn_selected=allocate_sn(
        csv_path=csv_path,   # 替换为实际路径
        per_total=per_total,               # 你要分配的总数量
        output_path='/home/franco/power/inverter1/inverter/code_v7/filter_sn/output.csv',
        seed=None  # 或者设一个固定值，如 42，以便结果可重复
    )
    return sn_selected
#根据得到的sn捞对应sn的数据
#timelist=['202409','202410','202411','202412']
#df: sn_prefix ┆ sn                   │
   
#def get_data_from_server(csv_path,time_list,per_total,seed):
    #df_select=get_train_sn(csv_path,per_total,seed)
def pick_20day_data_by_month(
    folder_path: str,
    timelist: list[str],
    df_select: pl.DataFrame,   # 里面有一列 'sn' 用于过滤
    output_folder: str,
    seed: int = None
):
    """
    从指定 folder_path 下的 ods_power_inverter_data_YYYYMMDD.parquet 文件中，
    针对 timelist 里的每个 YYYYMM，随机选取连续20天的数据并根据 df_select['sn'] 过滤，
    最终按月分别存到各自的子文件夹。
    
    :param folder_path: 存放原始 parquet 文件的文件夹路径。
    :param timelist: 需要处理的年月列表 (如 ["202409","202410","202411","202412"])。
    :param df_select: 一个 polars.DataFrame，其中至少包含一列 'sn'。只保留这些 SN 的数据。
    :param output_folder: 用于保存输出文件的文件夹。
    :param seed: （可选）随机种子，保证结果可重复。
    """
    if seed is not None:
        random.seed(seed)

    # 1. 列出文件夹下所有符合条件的 parquet 文件
    all_files = [
        f for f in os.listdir(folder_path)
        if f.endswith(".parquet") and f.startswith("ods_power_inverter_data_")
    ]

    # 2. 通过正则匹配从文件名中提取 YYYYMMDD
    #    假设文件名格式: ods_power_inverter_data_20240223.parquet
    pattern = r"ods_power_inverter_data_(\d{8}).parquet"

    # 用于按 {year_month: [(day_str, file_name), ...]} 方式存放文件信息
    files_by_ym = {}

    for f in all_files:
        match = re.match(pattern, f)
        if match:
            date_str = match.group(1)  # 例如 "20240223"
            year_month = date_str[:6]  # "202402"
            day_str = date_str[6:]     # "23"
            if year_month not in files_by_ym:
                files_by_ym[year_month] = []
            files_by_ym[year_month].append((day_str, f))

    # 3. 对每个 year_month 下的文件按 day_str 升序排序
    #    day_str 通常是 '01', '02' ... '31'，这里转整数再排序
    for ym in files_by_ym:
        files_by_ym[ym].sort(key=lambda x: int(x[0]))

    # 4. 从 df_select 提取 sn 列的所有值，供后续过滤（is_in）
    sn_list = df_select['sn'].unique().to_list()

    # 5. 针对 timelist 里的每个 YYYYMM，随机选取连续 20 天
    for ym in timelist:
        if ym not in files_by_ym:
            print(f"[警告] 文件夹中找不到 {ym} 相关的文件，跳过 ...")
            continue

        day_file_list = files_by_ym[ym]  # [(day_str, file_name), ...]

        # 如果当月文件天数 < 20，则无法抽取连续 20 天
        if len(day_file_list) < 20:
            print(f"[警告] {ym} 仅有 {len(day_file_list)} 天文件，不足 20 天，跳过 ...")
            continue

        # 随机选择一个起始下标，让其后续 19 天刚好拼够 20 天
        max_start = len(day_file_list) - 20
        start_idx = random.randint(0, max_start)

        # 取出连续20天 (start_idx 到 start_idx+19)
        chosen_20 = day_file_list[start_idx : start_idx + 20]

        # 6. 读这 20 天的文件，并根据 df_select['sn'] 做过滤，再合并
        frames = []
        for (day_str, file_name) in chosen_20:
            file_path = os.path.join(folder_path, file_name)
            df_day = pl.read_parquet(file_path)
            df_day_filtered = df_day.filter(pl.col('sn').is_in(sn_list))
            if df_day_filtered.height > 0:
                frames.append(df_day_filtered)

        if not frames:
            print(f"[提示] {ym} (20天范围) 内无匹配 sn，跳过保存。")
            continue
        all_columns=None
        month_20day_filtered = pl.concat(frames)
        month_20day_filtered,all_columns=DataProcessor(month_20day_filtered,all_columns).main1()
        month_20day_filtered = month_20day_filtered.with_columns(
            (pl.col("sn") + "_" + ym).alias("sn_ym")
        )

        # 7. 为该月创建单独的子文件夹，然后保存结果
        #    output_folder/202409, output_folder/202410, ...
        ym_folder = output_folder
        os.makedirs(ym_folder, exist_ok=True)

        # 写成 {YYYYMM}_20days_filtered.parquet
        output_file = f"{ym}_20days_filtered.parquet"
        output_path = os.path.join(ym_folder, output_file)
        month_20day_filtered.write_parquet(output_path)

        print(f"[完成] 为 {ym} 抽取连续20天并过滤 sn，共 {month_20day_filtered.height} 行，保存至 {output_path}")


# ============== 示例调用 ==============
if __name__ == "__main__":
    # 需要处理的年月列表
    timelist = ["202409","202410","202411","202412"]

    # 原始数据文件所在文件夹
    data_folder = "/home/franco/power/inverter/new_code/inverter_dataset/shang_neng/data_di/"
    #ok的sn
    csv_path="/home/franco/power/inverter1/inverter/code_v7/filter_sn/ok_sn.csv"

    # 输出结果保存的文件夹
    output_folder = "/data/inverter_power_data/processed_data/normal_data/"
    seed=None
    per_total=10000
    # 示例: 你已有一个 df_select，包含想要过滤的 sn
    df_select=get_train_sn(csv_path,per_total,seed)

    # 调用函数
    pick_20day_data_by_month(
        folder_path=data_folder,
        timelist=timelist,
        df_select=df_select,
        output_folder=output_folder,
        seed=seed # 可选：固定随机种子
    )









/tmp/ipykernel_23084/3230170342.py:34: DeprecationWarning: `groupby` is deprecated. It has been renamed to `group_by`.
  sn_counts = ok_sn.groupby('sn_prefix').agg(
/tmp/ipykernel_23084/3230170342.py:35: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
  pl.count().alias('count')
/tmp/ipykernel_23084/3230170342.py:66: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
  pl.when(pl.arange(0, pl.count()) < remainder)
/tmp/ipykernel_23084/3230170342.py:110: DeprecationWarning: `cumcount` is deprecated. It has been renamed to `cum_count`.
  pl.col('row_id').cumcount().over('sn_prefix').alias('row_num')


分配完成，每种 sn_prefix 分配的具体 sn 行如下：
shape: (10_000, 2)
┌───────────┬──────────────────────┐
│ sn_prefix ┆ sn                   │
│ ---       ┆ ---                  │
│ str       ┆ str                  │
╞═══════════╪══════════════════════╡
│ TP10KB    ┆ TP10KBT220530008     │
│ TP10KB    ┆ TP10KBT220530009     │
│ TP10KB    ┆ TP10KBT220530044     │
│ TP10KB    ┆ TP10KBT220530022     │
│ TP10KB    ┆ TP10KBT220614072     │
│ …         ┆ …                    │
│ TP30KB    ┆ TP30KBT000B230914006 │
│ TP30KB    ┆ TP30KBT000B230912007 │
│ TP25KB    ┆ TP25KBT000B240316130 │
│ TP20KB    ┆ TP20KBT000B240104135 │
│ TP25KB    ┆ TP25KBT000B231226239 │
└───────────┴──────────────────────┘
分配结果已保存到: /home/franco/power/inverter1/inverter/code_v7/filter_sn/output.csv
[完成] 为 202409 抽取连续20天并过滤 sn，共 24361102 行，保存至 /data/inverter_power_data/processed_data/normal_data/202409/202409_20days_filtered.parquet
[完成] 为 202410 抽取连续20天并过滤 sn，共 22747229 行，保存至 /data/inverter_power_data/processed_data/normal_data/202410/20

shape: (24_523_197,)
Series: 'sn_ym' [str]
[
	"TP40KBT000B240…
	"TP40KBT000B240…
	"TP40KBT000B240…
	"TP40KBT000B240…
	"TP40KBT000B240…
	"TP40KBT000B240…
	"TP40KBT000B240…
	"TP40KBT000B231…
	"TP40KBT000B240…
	"TP40KBT000B240…
	"TP40KBT000B240…
	"TP40KBT000B240…
	…
	"TP40KBT000B240…
	"TP40KBT000B240…
	"TP40KBT000B240…
	"TP40KBT000B240…
	"TP40KBT000B240…
	"TP40KBT000B240…
	"TP40KBT000B240…
	"TP40KBT000B240…
	"TP40KBT000B240…
	"TP40KBT000B240…
	"TP40KBT000B240…
	"TP40KBT000B240…
	"TP40KBT000B240…
]
唯一的 SN 数量: 8270
